# tiny-llm-fusion

The driver cells of the post, in its order. Run them from the repo root after `uv sync` and the exports in the README. Each section's cells use objects that earlier sections made, such as `dev`, `rec` and `every`.

## Sealed Benchmark Splits

In [ ]:
from pathlib import Path
from datasets import config
from fusion.data import SOURCES, load

tasks = {name: load(name) for name in SOURCES}
mb = sum(f.stat().st_size for f in Path(config.HF_DATASETS_CACHE).rglob("*") if f.is_file()) / 2**20
print(f"{sum(map(len, tasks.values())):,} rows from {len(tasks)} sources, {mb:,.0f} MB on disk")
t = tasks["gsm8k_train"][0]
print(t.id, "| gold", t.gold, "|", t.prompt)

In [ ]:
from fusion.data import draw, pools, seal

sealed, digest = seal([t for n in ("gsm8k", "math500", "mmlu", "humaneval_plus", "mbpp_plus", "bbh") for t in load(n)])
print(f"sealed {len(sealed):,} tasks, sha256 {digest[:12]}")
pool, dropped = pools(sealed, [t for n in list(SOURCES)[5:] for t in load(n)])   # from bbh on, its only split is test
print(f"pool {len(pool):,} rows, dropped {dropped['text']} repeated questions and {dropped['mbpp_id']} reused MBPP ids")
dev, train = draw(pool)
print(f"dev {len(dev):,} tasks, train {len(train):,} tasks")

## Serving Layer

In [ ]:
import threading
import time
import uvicorn
from fusion.config import CLOUD, MODELS
from fusion.platform.fleet import CloudClient, Reconciler, control
from fusion.platform.serve import boot_script

threading.Thread(target=uvicorn.run, args=(control,), kwargs={"host": "0.0.0.0", "port": 9000}, daemon=True).start()
rec = Reconciler(CloudClient(), boot=boot_script())
rec.tick({m: 1 for m in MODELS} | {"standby": 1}, idle=lambda v: False)
while any(v.state not in ("ready", "serving") for v in rec.vms):
    time.sleep(5)
    rec.refresh()
for v in rec.vms:
    print(f"{v.id}  {CLOUD['flavor']}  {v.model:<8}" + "".join(f"{s:>9} {t:>6.1f}s" for s, t in v.events))

## Inference Gateway

In [ ]:
from fusion.platform import gateway

gateway.sync(rec.vms)
print("registered", {m: [r.url for r in rs] for m, rs in gateway.REPLICAS.items()})

In [ ]:
import asyncio
import time
EASY, HARD = "What is 17 times 23?", "Prove that the sum of the first n odd numbers is n squared."

async def call(prompt, lane, tokens):
    body = {"model": "qwen", "messages": [{"role": "user", "content": prompt}], "max_tokens": tokens}
    t0 = time.perf_counter()
    await gateway.forward("/v1/chat/completions", body, lane)
    return time.perf_counter() - t0

async def main():
    for lane in ("single", "fanout"):
        hard = [asyncio.create_task(call(HARD, lane, 1024)) for _ in range(16 * 6)]
        await asyncio.sleep(1)
        easy = sorted(await asyncio.gather(*[call(EASY, "single", 16) for _ in range(40)]))
        done = max(await asyncio.gather(*hard))
        print(f"lane {lane:<7} easy p95 {easy[37]:5.2f} s   fan-out finished {done:6.1f} s")
await main()

## Sandboxed Execution and Grading

In [ ]:
from fusion.data import Task, load
from fusion.tools.graders import check, extract
from fusion.tools.sandbox import run

print(run("while True: pass"))
print(run("x = bytearray(2 * 1024 ** 3)").err.splitlines()[-1])
half = Task("demo/0", "math500", "math", "", r"\frac{1}{2}")
print(check(half, extract("math", r"so it is \boxed{0.5}")), check(half, extract("math", "so it is 0.5")))
task = load("humaneval_plus")[0]
code = task.prompt + "    return any(abs(a - b) < threshold for a, b in zip(numbers, numbers[1:]))\n"
print(run(code + "assert has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)").ok, check(task, code))

## Diverse Model Pool

In [ ]:
import asyncio
from fusion.engine.pool import POOL, ask, effective_voters, overlap
from fusion.tools.graders import check

out = {m: await asyncio.gather(*[ask(m, t, lane="batch") for t in dev]) for m in POOL}
right = {m: [int(check(t, c.answer)) for t, c in zip(dev, cands)] for m, cands in out.items()}
print({m: round(100 * sum(r) / len(dev), 1) for m, r in right.items()})
print(f"at least one model right: {100 * sum(map(any, zip(*right.values()))) / len(dev):.1f}%")
rho = sum(overlap(right).values()) / 6
print(f"mean rho of misses {rho:.2f}, four models vote like {effective_voters(4, rho):.2f}")

## Frontier Gap Diagnosis

In [ ]:
from fusion.data import load
from fusion.tools.frontier import live, score

t = load("gsm8k_train")[0]
msg, seconds = live(t)
print(msg.id, msg.stop_reason, [b.type for b in msg.content], msg.usage.input_tokens, msg.usage.output_tokens)
print(f"{seconds:.1f} s", score(t, msg))

In [ ]:
from fusion.tools.frontier import batch, score

FAMILIES, pct = ("math", "knowledge", "reasoning", "code"), lambda xs: 100 * sum(xs) / len(xs)
pilot = [t for f in FAMILIES for t in [t for t in dev if t.family == f][:200]]
opus, row = batch(pilot), {t.id: i for i, t in enumerate(dev)}
for f in FAMILIES:
    ts = [t for t in pilot if t.family == f]
    q = [right["qwen"][row[t.id]] for t in ts]
    o = [opus[t.id] is not None and check(t, score(t, opus[t.id])[0]) for t in ts]
    team = [any(right[m][row[t.id]] for m in POOL) for t in ts]
    print(f"{f:<10} qwen {pct(q):5.1f}   opus {pct(o):5.1f}   any model {pct(team):5.1f}")

## Program-Aided Reasoning Tool

In [ ]:
import asyncio
from fusion.tools.pal import pal_answer

t = load("gsm8k_train")[3]
cot, pot = await asyncio.gather(ask("llama", t), pal_answer("llama", t))
print("CoT", cot.answer, check(t, cot.answer), "| PoT", pot.answer, check(t, pot.answer))

## Knowledge Retrieval Tool

In [ ]:
from datasets import load_dataset
from fusion.engine.pool import PROMPT
from fusion.tools.retrieval import SKIP, augment, build_index

wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train")
build_index([" ".join(a["text"].split()[:120]) for a in wiki])
mmlu = [t for t in dev if t.bench == "mmlu"]
msgs = lambda t: [{"role": "user", "content": f"{augment(t)}\n\n{PROMPT[t.family]}"}]
rag = await asyncio.gather(*[ask("qwen", t, messages=msgs(t)) for t in mmlu])
print(f"{len(wiki):,} passages, {len(SKIP)} subjects skipped")
print(f"qwen on mmlu dev: {pct([right['qwen'][row[t.id]] for t in mmlu]):.1f} without, "
      f"{pct([check(t, c.answer) for t, c in zip(mmlu, rag)]):.1f} with")

## Parallel Candidate Sampling

In [ ]:
from fusion.engine.sampler import sample

t = dev[412]
cands = await sample(t, per_model=4, pal=2)
print(t.id)
for m in POOL:
    mine = [c for c in cands if c.model == m]
    print(f"{m:<6} {[c.answer for c in mine]}")

## Weighted Majority Voting

In [ ]:
from fusion.engine.vote import WEIGHT, tally

winner, margin, shares = tally(t, cands)
print(winner, round(margin, 3))
every = await asyncio.gather(*[sample(t) for t in dev])
for f in FAMILIES:
    ix = [i for i, t in enumerate(dev) if t.family == f]
    acc = lambda w: pct([check(dev[i], tally(dev[i], every[i], w)[0]) for i in ix])
    print(f"{f:<10} qwen {pct([right['qwen'][i] for i in ix]):5.1f}   majority {acc({m: 1.0 for m in POOL}):5.1f}"
          f"   weighted {acc(WEIGHT):5.1f}")

## Mixture of Agents Aggregation

In [ ]:
from fusion.engine.moa import moa

aggs = await asyncio.gather(*[moa(t, cs) for t, cs in zip(dev, every)])
base = pct([check(t, tally(t, cs)[0]) for t, cs in zip(dev, every)])
fused = pct([check(t, tally(t, cs + [a])[0]) for t, cs, a in zip(dev, every, aggs)])
print(f"vote {base:.1f}   vote with the aggregate {fused:.1f}")

## Verification and Calibration

In [ ]:
from fusion.engine.calibrate import ESCAPE, ece, fit
from fusion.engine.verifier import best_of_n

code = [i for i, t in enumerate(dev) if t.family == "code"]
picks = await asyncio.gather(*[best_of_n(dev[i], every[i] + [aggs[i]]) for i in code])
print(f"code with the verifier {pct([check(dev[i], c.answer) for i, (c, _) in zip(code, picks)]):.1f}")
won = [tally(t, cs + [a]) for t, cs, a in zip(dev, every, aggs)]
share, ok = [s.get(w, 0.0) for w, _, s in won], [check(t, w) for t, (w, _, _) in zip(dev, won)]
p = fit(share[::2], ok[::2]).predict(share[1::2])
print(f"ece {ece(share[1::2], ok[1::2]):.3f} raw, {ece(p, ok[1::2]):.3f} calibrated, {pct(p < ESCAPE):.1f}% below {ESCAPE}")

## Difficulty-Aware Router

In [ ]:
import numpy as np
from fusion.engine.router import COST, fit, probe, route

probes = await asyncio.gather(*[probe(t) for t in dev])
X = np.array([x for _, x in probes])
final = list(ok)
for i, (c, _) in zip(code, picks):
    final[i] = check(dev[i], c.answer)
tier = {1: right["qwen"], 2: [check(t, tally(t, cs[::2])[0]) for t, cs in zip(dev, every)], 3: final}
models = fit(X[::2], {t: np.array(v)[::2] for t, v in tier.items()})
plan, odd = [route(models, x) for x in X[1::2]], range(1, len(dev), 2)
print(f"routed {pct([tier[p][i] for p, i in zip(plan, odd)]):.1f} at {np.mean([COST[p] for p in plan]):.4f} credits, "
      f"always the full engine {pct(final[1::2]):.1f} at {COST[3]:.4f}")

## System Orchestrator

In [ ]:
import uvicorn
from openai import AsyncOpenAI
from fusion.engine.calibrate import fit as fit_calibrator
from fusion.serve_app import orchestrator

orchestrator.ENGINE = orchestrator.FusionEngine(models, fit_calibrator(share, ok))
asyncio.create_task(uvicorn.Server(uvicorn.Config(orchestrator.API, port=8080)).serve())
client = AsyncOpenAI(base_url="http://127.0.0.1:8080/v1", api_key="unused")
r = await client.chat.completions.create(model="fusion-4x3b", extra_body={"family": "math"},
                                         messages=[{"role": "user", "content": dev[412].prompt}])
print(r.choices[0].message.content, r.model_extra["fusion"])

## Self-Improvement Loop

In [ ]:
from fusion.platform import episodes

orchestrator.ENGINE.store = True
await orchestrator.ENGINE.run_stream(train)
print(episodes.DB.execute("SELECT COUNT(*), SUM(ok), COUNT(DISTINCT task) FROM ep").fetchone())

In [ ]:
from fusion.learn.improve import hot_load, mine_pairs

for m in ("llama", "gemma"):
    hot_load(m, f"fusion-team/{m}-dpo")
    after = await asyncio.gather(*[ask(m, t, lane="batch") for t in dev])
    print(f"{m:<6} {len(mine_pairs(m)):,} pairs   dev {pct(right[m]):.1f} -> {pct([check(t, c.answer) for t, c in zip(dev, after)]):.1f}")

## Evaluation Harness

In [ ]:
from fusion.eval.harness import ACCESS, gap_closed, grade, interval

BENCHES = ["gsm8k", "math500", "mmlu", "humaneval_plus", "mbpp_plus", "bbh"]
paired = [t for b in BENCHES for t in [t for t in sealed if t.bench == b][:400]]
runs = {"best_single": [a for a, _ in await asyncio.gather(*map(probe, paired))],
        "fused_dpo": [a for a, _ in await orchestrator.ENGINE.run_stream(paired)]}
gateway.ADAPTER.clear()
runs["fused"] = [a for a, _ in await orchestrator.ENGINE.run_stream(paired)]
opus = batch(paired)
runs["opus"] = [score(t, opus[t.id])[0] if opus[t.id] else None for t in paired]
graded = {arm: grade(arm, paired, answers) for arm, answers in runs.items()}
print(f"{len(paired):,} paired tasks, {len(ACCESS):,} reads logged, opus refusals "
      f"{sum(m is not None and m.stop_reason == 'refusal' for m in opus.values())}")

## Accuracy Against Spend

In [ ]:
acc = {a: {b: 100 * r[[t.bench == b for t in paired]].mean() for b in BENCHES} for a, r in graded.items()}
for a in acc:
    acc[a]["mean"] = np.mean([acc[a][b] for b in BENCHES])
ARMS = ("best_single", "fused", "fused_dpo", "opus")
print(f"{'':<15}" + "".join(f"{a:>12}" for a in ARMS) + "   gap closed")
for b in BENCHES + ["mean"]:
    row = [acc[a][b] for a in ARMS]
    print(f"{b:<15}" + "".join(f"{x:12.1f}" for x in row) + f"   {gap_closed(row[0], row[2], row[3]):10.2f}")

## Ablation Study

In [ ]:
macro = lambda r: np.mean([r[[t.bench == b for t in paired]].mean() for b in BENCHES])
ref = orchestrator.FusionEngine(models, orchestrator.ENGINE.cal)
base = macro(grade("full", paired, [a for a, _ in await ref.run_stream(paired)]))
for row in ("router", "diversity", "pot", "retrieval", "moa", "weights", "tests", "dpo"):
    if row == "dpo":
        gateway.ADAPTER.clear()
    engine = orchestrator.FusionEngine(models, orchestrator.ENGINE.cal, off={row})
    out = macro(grade(row, paired, [a for a, _ in await engine.run_stream(paired)]))
    print(f"{row:<10} {100 * (out - base):+6.2f} points {engine.gov.spent / ref.gov.spent:5.2f}x credits")